In [2]:
from ultralytics import YOLO

In [3]:
model = YOLO("yolov8n.pt")

In [4]:
import cv2

cap = cv2.VideoCapture("test.mp4")

print("Video opened:", cap.isOpened())

Video opened: True


In [5]:
FOCAL_LENGTH = 700
KNOWN_WIDTH = 1.8

previous_distance = None
previous_time = None
smoothed_ttc = None

def estimate_distance(box_width):
    box_width = max(box_width, 1)
    return (KNOWN_WIDTH * FOCAL_LENGTH) / box_width

In [6]:
from ultralytics import YOLO
import cv2
import time

# Load YOLO model
model = YOLO("yolov8n.pt")

# Open input video
cap = cv2.VideoCapture("test.mp4")

FOCAL_LENGTH = 700
KNOWN_WIDTH = 1.8

previous_distance = None
previous_time = None
smoothed_ttc = None


def estimate_distance(box_width):
    box_width = max(box_width, 1)
    return (KNOWN_WIDTH * FOCAL_LENGTH) / box_width


while True:
    ret, frame = cap.read()

    if not ret:
        break

    current_time = time.time()

    # Detect objects in current frame
    results = model(frame, verbose=False)[0]

    best_distance = None
    best_box = None

    # Find closest vehicle
    for box in results.boxes:

        name = model.names[int(box.cls[0])]

        if name not in ["car", "truck", "bus", "motorcycle"]:
            continue

        x1, y1, x2, y2 = map(int, box.xyxy[0])

        width = x2 - x1
        distance = estimate_distance(width)

        if best_distance is None or distance < best_distance:
            best_distance = distance
            best_box = (x1, y1, x2, y2)

    ttc = float("inf")

    if best_distance is not None:

        if previous_distance is not None and previous_time is not None:

            dt = max(current_time - previous_time, 1e-3)

            relative_speed = (
                previous_distance - best_distance
            ) / dt

            if relative_speed > 0.1:

                ttc = best_distance / relative_speed

                if smoothed_ttc is None:
                    smoothed_ttc = ttc
                else:
                    smoothed_ttc = (
                        0.7 * smoothed_ttc +
                        0.3 * ttc
                    )

        previous_distance = best_distance
        previous_time = current_time

        # Draw vehicle box
        if best_box:

            x1, y1, x2, y2 = best_box

            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                (0, 255, 255),
                2
            )

            cv2.putText(
                frame,
                f"Distance: {best_distance:.1f} m",
                (x1, max(y1 - 10, 20)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (255, 255, 0),
                2
            )

    displayed_ttc = (
        smoothed_ttc
        if smoothed_ttc is not None
        else float("inf")
    )

    # Warning levels
    if displayed_ttc < 2:

        warning = "CRITICAL: COLLISION WARNING!"
        warning_color = (0, 0, 255)

    elif displayed_ttc < 4:

        warning = "CAUTION: REDUCE SPEED"
        warning_color = (0, 165, 255)

    else:

        warning = "SAFE"
        warning_color = (0, 255, 0)

    # Display warning
    cv2.putText(
        frame,
        warning,
        (25, 45),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.9,
        warning_color,
        3
    )

    # Display TTC
    cv2.putText(
        frame,
        f"TTC: {displayed_ttc:.2f} s",
        (25, 85),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    # Show output
    cv2.imshow(
        "Forward Collision Warning",
        frame
    )

    # Press q to stop
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


cap.release()
cv2.destroyAllWindows()